# Module 1: Static Factor Model Basics

## Learning Objectives

By completing this notebook, you will:

1. **Formulate** the static factor model in matrix notation
2. **Simulate** factor model data with controlled properties
3. **Extract** factors using Principal Component Analysis
4. **Interpret** factor loadings and assess model fit
5. **Understand** the identification problem and rotational indeterminacy

## Prerequisites

- Module 0 (matrix algebra, PCA fundamentals)
- Understanding of covariance matrices and eigendecomposition

## Estimated Time: 90-120 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "the static factor model in matrix notation",
    "factor model data with controlled properties",
    "factors using Principal Component Analysis",
    "factor loadings and assess model fit",
    "the identification problem and rotational indeterminacy"
])

## Setup and Imports

We'll build on the PCA implementation from Module 0 and add factor model-specific functionality.

In [ ]:
# Purpose: Import libraries for factor model simulation and estimation
# Key Concept: Factor models combine linear algebra with statistical modeling

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

---

# Part 1: Static Factor Model Specification

## 1.1 The Model

The static factor model decomposes observed variables into common and idiosyncratic components:

$$X_t = \Lambda F_t + e_t$$

where:
- $X_t$: $N \times 1$ vector of observed variables at time $t$
- $F_t$: $r \times 1$ vector of latent factors (common shocks)
- $\Lambda$: $N \times r$ matrix of factor loadings (sensitivities)
- $e_t$: $N \times 1$ vector of idiosyncratic errors

**Key assumptions:**
1. $E[F_t] = 0$, $E[F_t F_t'] = I_r$ (factors are normalized)
2. $E[e_t] = 0$, $E[e_t e_t'] = \Sigma_e$ (idiosyncratic covariance)
3. $E[F_t e_t'] = 0$ (factors uncorrelated with idiosyncratic errors)

**Implied covariance structure:**
$$\Sigma_X = \Lambda \Lambda' + \Sigma_e$$

## 1.2 Simulating Factor Model Data

To understand factor models, we first simulate data where we know the true factor structure.

In [ ]:
# Purpose: Simulate data from a static factor model
# Key Concept: Generate X = Lambda * F + e with known parameters

def simulate_static_factor_model(T, N, r, loading_scale=1.0, 
                                  idiosyncratic_std=0.5, seed=None):
    """
    Simulate data from a static factor model.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Number of observed variables
    r : int
        Number of factors
    loading_scale : float
        Scale parameter for loadings (controls factor strength)
    idiosyncratic_std : float
        Standard deviation of idiosyncratic errors
    seed : int, optional
        Random seed
    
    Returns
    -------
    X : ndarray, shape (T, N)
        Observed data
    F_true : ndarray, shape (T, r)
        True factors
    Lambda_true : ndarray, shape (N, r)
        True factor loadings
    """
    if seed is not None:
        np.random.seed(seed)
    
    # Step 1: Generate factors (i.i.d. standard normal)
    F_true = np.random.randn(T, r)
    
    # Step 2: Generate loadings (random with specified scale)
    Lambda_true = np.random.randn(N, r) * loading_scale
    
    # Step 3: Generate idiosyncratic errors
    e = np.random.randn(T, N) * idiosyncratic_std
    
    # Step 4: Construct observed data
    X = F_true @ Lambda_true.T + e
    
    return X, F_true, Lambda_true


# Simulate a simple factor model
T, N, r = 200, 10, 2
X, F_true, Lambda_true = simulate_static_factor_model(
    T=T, N=N, r=r, 
    loading_scale=1.0, 
    idiosyncratic_std=0.5, 
    seed=123
)

print(f"Simulated factor model data:")
print(f"  Time periods (T): {T}")
print(f"  Variables (N): {N}")
print(f"  Factors (r): {r}")
print(f"\nData shape: {X.shape}")
print(f"True factors shape: {F_true.shape}")
print(f"True loadings shape: {Lambda_true.shape}")

# Verify covariance structure
Sigma_X_sample = np.cov(X.T)
Sigma_X_implied = Lambda_true @ Lambda_true.T + np.eye(N) * 0.5**2

print(f"\nCovariance structure check:")
print(f"  Sample covariance trace: {np.trace(Sigma_X_sample):.3f}")
print(f"  Implied covariance trace: {np.trace(Sigma_X_implied):.3f}")

### Visualize the Data and True Factors

In [ ]:
# Purpose: Visualize simulated data and true factor structure
# Key Concept: Factor models capture co-movement across variables

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: First 4 observed variables
for i in range(4):
    axes[0, 0].plot(X[:, i], alpha=0.7, label=f'Var {i+1}')
axes[0, 0].set_title('Sample of Observed Variables')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Value')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: True factors
axes[0, 1].plot(F_true[:, 0], linewidth=2, label='Factor 1', color='steelblue')
axes[0, 1].plot(F_true[:, 1], linewidth=2, label='Factor 2', color='coral')
axes[0, 1].set_title('True Latent Factors')
axes[0, 1].set_xlabel('Time')
axes[0, 1].set_ylabel('Factor Value')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: True loadings
var_names = [f'V{i+1}' for i in range(N)]
x_pos = np.arange(N)
width = 0.35
axes[1, 0].bar(x_pos - width/2, Lambda_true[:, 0], width, 
               label='Factor 1', alpha=0.8, color='steelblue')
axes[1, 0].bar(x_pos + width/2, Lambda_true[:, 1], width, 
               label='Factor 2', alpha=0.8, color='coral')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(var_names)
axes[1, 0].set_title('True Factor Loadings')
axes[1, 0].set_ylabel('Loading')
axes[1, 0].legend()
axes[1, 0].axhline(y=0, color='black', linewidth=0.5)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Correlation matrix
corr_matrix = np.corrcoef(X.T)
im = axes[1, 1].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1, 1].set_title('Correlation Matrix of Observed Variables')
axes[1, 1].set_xticks(range(N))
axes[1, 1].set_yticks(range(N))
axes[1, 1].set_xticklabels(var_names)
axes[1, 1].set_yticklabels(var_names)
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print("\nNote: Variables co-move because they respond to common factors!")

### Exercise 1.1: Understanding Factor Strength

**Task:** Simulate data with different factor strengths and observe how it affects the correlation structure.

**Expected Output:** Stronger factors (higher loading_scale) should produce higher correlations among variables.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Simulate three datasets with different factor strengths
#       and compare average off-diagonal correlations

def analyze_factor_strength(loading_scale):
    """
    Simulate data and compute average correlation.
    
    Parameters
    ----------
    loading_scale : float
        Scale of factor loadings (factor strength)
    
    Returns
    -------
    avg_corr : float
        Average absolute off-diagonal correlation
    """
    # Simulate data
    X, _, _ = None  # Replace with simulate_static_factor_model call
    
    # Compute correlation matrix
    corr_matrix = None  # Replace with np.corrcoef(X.T)
    
    # Get off-diagonal elements (set diagonal to 0)
    np.fill_diagonal(corr_matrix, 0)
    
    # Average absolute correlation
    avg_corr = None  # Replace with np.abs(corr_matrix).mean()
    
    return avg_corr


# Test different factor strengths
strengths = [0.5, 1.0, 2.0]
avg_correlations = []

for scale in strengths:
    avg_corr = analyze_factor_strength(scale)
    avg_correlations.append(avg_corr)
    print(f"Loading scale {scale:.1f}: Avg correlation = {avg_corr:.3f}")

# Visualize relationship
plt.figure(figsize=(8, 5))
plt.plot(strengths, avg_correlations, 'o-', linewidth=2, markersize=10)
plt.xlabel('Factor Loading Scale')
plt.ylabel('Average Absolute Correlation')
plt.title('Factor Strength vs Cross-Sectional Correlation')
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>Hint 1: Simulation</summary>
Use: X, _, _ = simulate_static_factor_model(T=200, N=10, r=2, loading_scale=loading_scale, seed=42)
</details>

<details>
<summary>Hint 2: Average Correlation</summary>
After setting diagonal to 0, use: avg_corr = np.abs(corr_matrix).mean()
</details>

In [ ]:
# AUTO-GRADED TESTS
# ----------------------------------
def test_exercise_1_1():
    corr_weak = analyze_factor_strength(0.5)
    corr_medium = analyze_factor_strength(1.0)
    corr_strong = analyze_factor_strength(2.0)
    
    assert corr_weak is not None, "Don't forget to compute average correlation!"
    assert 0 <= corr_weak <= 1, "Correlation should be between 0 and 1"
    assert corr_weak < corr_medium < corr_strong, "Stronger factors should increase correlation"
    
    print("✓ Exercise 1.1 passed! Factor strength correctly affects correlations.")

test_exercise_1_1()

In [ ]:
# SOLUTION
# --------
def analyze_factor_strength_solution(loading_scale):
    X, _, _ = simulate_static_factor_model(T=200, N=10, r=2, 
                                           loading_scale=loading_scale, seed=42)
    corr_matrix = np.corrcoef(X.T)
    np.fill_diagonal(corr_matrix, 0)
    avg_corr = np.abs(corr_matrix).mean()
    return avg_corr

---

# Part 2: Factor Extraction via PCA

## 2.1 PCA as Factor Estimator

Under the static factor model, the **principal components** provide consistent estimates of the factors (up to rotation) when $N$ is large.

**Stock-Watson approach:**
1. Standardize each variable (optional but common)
2. Compute principal components of the data
3. The first $r$ PCs estimate the $r$ factors
4. PC loadings estimate factor loadings

In [ ]:
# Purpose: Extract factors using PCA and compare to true factors
# Key Concept: PCA recovers factors up to rotation and scale

def extract_factors_pca(X, n_factors, standardize=False):
    """
    Extract factors using Principal Component Analysis.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Data matrix
    n_factors : int
        Number of factors to extract
    standardize : bool
        Whether to standardize variables before PCA
    
    Returns
    -------
    F_hat : ndarray, shape (T, n_factors)
        Estimated factors (PC scores)
    Lambda_hat : ndarray, shape (N, n_factors)
        Estimated loadings (PC loadings)
    explained_var : ndarray, shape (n_factors,)
        Variance explained by each factor
    """
    T, N = X.shape
    
    # Step 1: Optionally standardize
    if standardize:
        X_processed = (X - X.mean(axis=0)) / X.std(axis=0)
    else:
        X_processed = X - X.mean(axis=0)
    
    # Step 2: Run PCA
    pca = PCA(n_components=n_factors)
    F_hat = pca.fit_transform(X_processed)
    
    # Step 3: Extract loadings
    Lambda_hat = pca.components_.T  # Transpose to get N x r
    
    # Step 4: Get explained variance
    explained_var = pca.explained_variance_ratio_
    
    return F_hat, Lambda_hat, explained_var


# Extract factors from simulated data
F_hat, Lambda_hat, explained_var = extract_factors_pca(X, n_factors=r)

print("Factor Extraction Results:")
print(f"\nEstimated factors shape: {F_hat.shape}")
print(f"Estimated loadings shape: {Lambda_hat.shape}")
print(f"\nVariance explained:")
for i, var in enumerate(explained_var, 1):
    print(f"  Factor {i}: {var*100:.2f}%")
print(f"  Total: {explained_var.sum()*100:.2f}%")

## 2.2 Comparing Estimated and True Factors

**Important:** Factors are only identified up to rotation and sign. We need to align estimated factors with true factors using correlation.

In [ ]:
callout("** Factors are only identified up to rotation and sign. We need to align estimated factors with true factors using correlation.", "warning")

In [ ]:
# Purpose: Compare estimated factors with true factors
# Key Concept: Factors identified up to rotation - use correlation to match

def align_factors(F_true, F_hat):
    """
    Align estimated factors with true factors using correlation.
    
    Finds best match by maximizing correlation and flipping signs if needed.
    
    Parameters
    ----------
    F_true : ndarray, shape (T, r)
        True factors
    F_hat : ndarray, shape (T, r)
        Estimated factors
    
    Returns
    -------
    F_hat_aligned : ndarray, shape (T, r)
        Aligned estimated factors
    correlations : ndarray, shape (r,)
        Correlation of each aligned factor with truth
    """
    r = F_true.shape[1]
    F_hat_aligned = F_hat.copy()
    correlations = np.zeros(r)
    
    # Compute correlation matrix between true and estimated
    corr_matrix = np.corrcoef(F_true.T, F_hat.T)[:r, r:]
    
    # Greedy matching: assign each estimated factor to best true factor
    used_true = []
    for i in range(r):
        # Find best match among unused true factors
        abs_corr = np.abs(corr_matrix[:, i])
        best_match = np.argmax(abs_corr)
        
        correlations[best_match] = corr_matrix[best_match, i]
        
        # Flip sign if negatively correlated
        if correlations[best_match] < 0:
            F_hat_aligned[:, i] = -F_hat_aligned[:, i]
            correlations[best_match] = -correlations[best_match]
    
    return F_hat_aligned, correlations


# Align and compare
F_hat_aligned, correlations = align_factors(F_true, F_hat)

print("Factor Alignment Results:")
for i, corr in enumerate(correlations, 1):
    print(f"  Factor {i}: correlation = {corr:.4f}")

# Visualize comparison
fig, axes = plt.subplots(r, 1, figsize=(12, 4*r))
if r == 1:
    axes = [axes]

for i in range(r):
    axes[i].plot(F_true[:, i], linewidth=2, label='True Factor', alpha=0.8)
    axes[i].plot(F_hat_aligned[:, i], linewidth=2, label='Estimated Factor (PCA)', 
                 alpha=0.8, linestyle='--')
    axes[i].set_title(f'Factor {i+1} Comparison (Correlation: {correlations[i]:.4f})')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('Factor Value')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Exercise 2.1: Factor Recovery Quality

**Task:** Investigate how factor recovery quality depends on the signal-to-noise ratio.

**Theory:** As idiosyncratic noise increases, factor estimates become less accurate.

**Expected Output:** Higher noise should lead to lower correlations between true and estimated factors.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Compute factor recovery correlation for different noise levels

def factor_recovery_quality(idiosyncratic_std):
    """
    Assess factor recovery quality for given noise level.
    
    Parameters
    ----------
    idiosyncratic_std : float
        Standard deviation of idiosyncratic errors
    
    Returns
    -------
    avg_correlation : float
        Average correlation between true and estimated factors
    """
    # Simulate data with specified noise level
    X_test, F_test, _ = None  # Replace with simulate_static_factor_model call
    
    # Extract factors
    F_hat_test, _, _ = None  # Replace with extract_factors_pca call
    
    # Align and compute correlations
    _, correlations = None  # Replace with align_factors(F_test, F_hat_test)
    
    # Return average correlation
    avg_correlation = None  # Replace with correlations.mean()
    
    return avg_correlation


# Test different noise levels
noise_levels = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5]
recovery_quality = []

for noise in noise_levels:
    quality = factor_recovery_quality(noise)
    recovery_quality.append(quality)
    print(f"Noise std {noise:.1f}: Avg factor correlation = {quality:.3f}")

# Plot results
plt.figure(figsize=(8, 5))
plt.plot(noise_levels, recovery_quality, 'o-', linewidth=2, markersize=10)
plt.xlabel('Idiosyncratic Noise (Std Dev)')
plt.ylabel('Average Factor Recovery Correlation')
plt.title('Factor Recovery Quality vs Noise Level')
plt.grid(True, alpha=0.3)
plt.show()

<details>
<summary>Hint 1: Simulation</summary>
Use: X_test, F_test, _ = simulate_static_factor_model(T=200, N=10, r=2, idiosyncratic_std=idiosyncratic_std, seed=42)
</details>

<details>
<summary>Hint 2: Alignment</summary>
Use: _, correlations = align_factors(F_test, F_hat_test)
Then: avg_correlation = correlations.mean()
</details>

In [ ]:
# AUTO-GRADED TESTS
# ----------------------------------
def test_exercise_2_1():
    quality_low = factor_recovery_quality(0.1)
    quality_med = factor_recovery_quality(0.5)
    quality_high = factor_recovery_quality(1.5)
    
    assert quality_low is not None, "Don't forget to compute recovery quality!"
    assert 0 <= quality_low <= 1, "Correlation should be between 0 and 1"
    assert quality_low > quality_med > quality_high, "Higher noise should reduce recovery quality"
    assert quality_low > 0.8, "With low noise, recovery should be very good (>0.8)"
    
    print("✓ Exercise 2.1 passed! Factor recovery quality computed correctly.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
def factor_recovery_quality_solution(idiosyncratic_std):
    X_test, F_test, _ = simulate_static_factor_model(
        T=200, N=10, r=2, idiosyncratic_std=idiosyncratic_std, seed=42
    )
    F_hat_test, _, _ = extract_factors_pca(X_test, n_factors=2)
    _, correlations = align_factors(F_test, F_hat_test)
    return correlations.mean()

---

# Part 3: Loading Interpretation and Model Diagnostics

## 3.1 Interpreting Factor Loadings

Loadings $\lambda_{ij}$ measure how variable $i$ responds to factor $j$:
- **Large positive:** Variable increases when factor increases
- **Large negative:** Variable decreases when factor increases
- **Near zero:** Variable unrelated to that factor

In [ ]:
# Purpose: Create interpretable factor structure (macro-style)
# Key Concept: Loadings reveal which variables respond to which factors

# Simulate data with interpretable structure
T = 250
N = 12
r = 2

# Design loadings with economic interpretation
# Factor 1: "Real activity" - affects production, employment, sales
# Factor 2: "Inflation" - affects prices, wages, interest rates
Lambda_designed = np.array([
    [0.95, 0.10],   # Industrial production
    [0.90, 0.15],   # Manufacturing output
    [0.85, 0.20],   # Employment
    [0.80, 0.15],   # Retail sales
    [0.75, 0.25],   # Personal income
    [0.70, 0.30],   # Housing starts
    [0.20, 0.90],   # CPI inflation
    [0.15, 0.88],   # PPI inflation
    [0.25, 0.85],   # Wage growth
    [0.30, 0.80],   # Import prices
    [0.55, 0.60],   # Interest rate (responds to both)
    [0.50, 0.55],   # Exchange rate (responds to both)
])

# Generate factors and data
np.random.seed(789)
F_designed = np.random.randn(T, r)
e_designed = np.random.randn(T, N) * 0.3
X_designed = F_designed @ Lambda_designed.T + e_designed

# Variable names
var_names_full = [
    'IP', 'Mfg Output', 'Employment', 'Retail Sales', 'Pers Income', 'Housing',
    'CPI', 'PPI', 'Wage Growth', 'Import Prices', 'Interest Rate', 'Exch Rate'
]

# Extract factors
F_hat_designed, Lambda_hat_designed, explained_var_designed = extract_factors_pca(
    X_designed, n_factors=r
)

print(f"Designed macro panel:")
print(f"  Variables: {N}")
print(f"  Factors: {r}")
print(f"  Variance explained: {explained_var_designed.sum()*100:.1f}%")

In [ ]:
# Purpose: Visualize and interpret estimated loadings
# Key Concept: Loadings reveal factor interpretation

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Factor 1 loadings
axes[0].barh(var_names_full, Lambda_hat_designed[:, 0], alpha=0.8, color='steelblue')
axes[0].set_xlabel('Loading Value')
axes[0].set_title(f'Factor 1 Loadings\n(Explains {explained_var_designed[0]*100:.1f}% of variance)')
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].grid(True, alpha=0.3, axis='x')

# Factor 2 loadings
axes[1].barh(var_names_full, Lambda_hat_designed[:, 1], alpha=0.8, color='coral')
axes[1].set_xlabel('Loading Value')
axes[1].set_title(f'Factor 2 Loadings\n(Explains {explained_var_designed[1]*100:.1f}% of variance)')
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Factor 1: Real Activity (high loadings on production, employment, sales)")
print("  Factor 2: Inflation (high loadings on price indices, wages)")
print("  Interest rate and exchange rate respond to both factors.")

## 3.2 Model Fit Assessment

Two key diagnostics:
1. **Variance explained:** What proportion of total variance do factors capture?
2. **Communalities:** For each variable, what proportion of its variance comes from common factors?

In [ ]:
# Purpose: Compute model diagnostics
# Key Concept: Assess how well factors explain observed variation

def compute_model_diagnostics(X, F_hat, Lambda_hat):
    """
    Compute factor model fit diagnostics.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Original data
    F_hat : ndarray, shape (T, r)
        Estimated factors
    Lambda_hat : ndarray, shape (N, r)
        Estimated loadings
    
    Returns
    -------
    communalities : ndarray, shape (N,)
        Proportion of variance explained by factors for each variable
    uniqueness : ndarray, shape (N,)
        Proportion of variance that is idiosyncratic
    """
    T, N = X.shape
    
    # Center data
    X_centered = X - X.mean(axis=0)
    
    # Reconstruct common component
    X_common = F_hat @ Lambda_hat.T
    
    # Compute communalities (R-squared for each variable)
    communalities = np.zeros(N)
    uniqueness = np.zeros(N)
    
    for i in range(N):
        # Total variance
        total_var = np.var(X_centered[:, i])
        
        # Variance explained by factors
        common_var = np.var(X_common[:, i])
        
        communalities[i] = common_var / total_var
        uniqueness[i] = 1 - communalities[i]
    
    return communalities, uniqueness


# Compute diagnostics
communalities, uniqueness = compute_model_diagnostics(
    X_designed, F_hat_designed, Lambda_hat_designed
)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Communalities
axes[0].barh(var_names_full, communalities, alpha=0.8, color='forestgreen')
axes[0].set_xlabel('Communality (R²)')
axes[0].set_title('Proportion of Variance Explained by Factors')
axes[0].set_xlim([0, 1])
axes[0].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='x')

# Uniqueness
axes[1].barh(var_names_full, uniqueness, alpha=0.8, color='indianred')
axes[1].set_xlabel('Uniqueness (1 - R²)')
axes[1].set_title('Proportion of Idiosyncratic Variance')
axes[1].set_xlim([0, 1])
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nSummary Statistics:")
print(f"  Average communality: {communalities.mean():.3f}")
print(f"  Min communality: {communalities.min():.3f} ({var_names_full[communalities.argmin()]})")
print(f"  Max communality: {communalities.max():.3f} ({var_names_full[communalities.argmax()]})")

---

# Summary and Key Takeaways

## What You've Learned

1. **Static Factor Model Specification**
   - $X_t = \Lambda F_t + e_t$ decomposes data into common and idiosyncratic components
   - Covariance structure: $\Sigma_X = \Lambda\Lambda' + \Sigma_e$
   - Factor strength determines cross-sectional correlation

2. **Factor Extraction via PCA**
   - Principal components consistently estimate factors (as $N \to \infty$)
   - Factors identified up to rotation and sign
   - Recovery quality depends on signal-to-noise ratio

3. **Interpretation and Diagnostics**
   - Loadings reveal which variables respond to which factors
   - Communalities measure fit for each variable
   - Economic interpretation: real activity, inflation, financial conditions

## Connection to Next Modules

| Next Module | Connection |
|-------------|------------|
| Module 2: Dynamic Factors | Add time series dynamics: $F_t = \Phi F_{t-1} + \eta_t$ |
| Module 3: PCA Estimation | Formal large-N theory, factor number selection |
| Module 4: ML Estimation | Maximum likelihood estimation via Kalman filter |

## Key Concepts to Remember

1. **Identification:** Factors are unique only up to rotation $\Lambda H$ and $H^{-1}F$
2. **Consistency:** PCA estimates converge to true factors as $N \to \infty$
3. **Interpretation:** Examine loadings to understand factor meaning
4. **Fit:** High communalities indicate strong factor structure

## Next Steps

After completing this notebook:
1. Review the identification guide (`guides/02_identification_problem.md`)
2. Learn about approximate factor models (`guides/03_approximate_factor_models.md`)
3. Proceed to Module 2: Dynamic Factor Models

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Simulate data from a static factor model
- [ ] Extract factors using PCA
- [ ] Align estimated factors with true factors
- [ ] Interpret factor loadings economically
- [ ] Compute and interpret communalities
- [ ] Explain why factors are only identified up to rotation

---

*Well done! You now understand the static factor model and can extract factors using PCA. In Module 2, we'll add dynamics to create true dynamic factor models.*